In [0]:
# Load all 3 months
jan = spark.read.csv(
    "/Volumes/workspace/default/bts_practice_data/January_2024.csv",
    header=True, inferSchema=True)

feb = spark.read.csv(
    "/Volumes/workspace/default/bts_practice_data/February_2024.csv",
    header=True, inferSchema=True)

mar = spark.read.csv(
    "/Volumes/workspace/default/bts_practice_data/March_2024.csv",
    header=True, inferSchema=True)

df = jan.unionByName(feb).unionByName(mar)
print(f"Total rows: {df.count():,}")

In [0]:
df.groupBy("MONTH").count().orderBy("MONTH").show()

**Day_11**

In [0]:
# Create carrier lookup
carrier_data = [
    ("AA", "American Airlines"), ("DL", "Delta Air Lines"),
    ("UA", "United Airlines"), ("WN", "Southwest Airlines"),
    ("B6", "JetBlue"), ("AS", "Alaska Airlines"),
    ("NK", "Spirit Airlines"), ("F9", "Frontier Airlines"),
    ("9E", "Endeavor Air"), ("OO", "SkyWest Airlines"),
    ("MQ", "Envoy Air"), ("OH", "PSA Airlines"),
    ("HA", "Hawaiian Airlines"), ("G4", "Allegiant Air"),
    ("YX", "Republic Airways")
]
schema = ["CARRIER_CODE", "CARRIER_NAME"]
carrier_lookup = spark.createDataFrame(carrier_data, schema)

# Inner Join
from pyspark.sql.functions import broadcast
result = df.join(broadcast(carrier_lookup),
                 df["OP_UNIQUE_CARRIER"] == carrier_lookup["CARRIER_CODE"],
                 "inner")
result.select("CARRIER_NAME", "ORIGIN", "DEST", "DEP_DELAY").show(5)
print(f"Total rows: {result.count():,}")

In [0]:
# Find unmatched carriers
small_lookup = spark.createDataFrame(
    [("AA", "American Airlines"), ("DL", "Delta Air Lines")],
    ["CARRIER_CODE", "CARRIER_NAME"]
)

result = df.join(broadcast(small_lookup),
                 df["OP_UNIQUE_CARRIER"] == small_lookup["CARRIER_CODE"],
                 "left")

unmatched = result.filter(result["CARRIER_NAME"].isNull())
unmatched.select("OP_UNIQUE_CARRIER").distinct().show()
print(f"Unmatched: {unmatched.select('OP_UNIQUE_CARRIER').distinct().count()}")

In [0]:
result = df.filter(df["MONTH"] == 1) \
           .join(broadcast(carrier_lookup),
                 df["OP_UNIQUE_CARRIER"] == carrier_lookup["CARRIER_CODE"],
                 "inner") \
           .groupBy("CARRIER_NAME", "ORIGIN_STATE_ABR") \
           .count() \
           .orderBy("count", ascending=False)

result.show(10)

**Day_12**

In [0]:
# Create a UDF that categorizes DEP_DELAY:
#   → NULL   → "CANCELLED"
#   → <= 0   → "ON_TIME"
#   → <= 30  → "MINOR"
#   → > 30   → "MAJOR"
#
# Register as UDF and apply to DEP_DELAY column.
# GroupBy DELAY_CATEGORY and count.
# Sort descending.

from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

def categorize(delay):
    if delay is None: return "CANCELLED"
    if delay <= 0: return "ON_TIME"
    if delay <= 30: return "MINOR"
    return "MAJOR"

categorize_udf = udf(categorize, StringType())

ps32 = df.withColumn("DELAY_CATEGORY", categorize_udf(col("DEP_DELAY")))
ps32.groupBy("DELAY_CATEGORY").count().orderBy("count", ascending=False).show()

In [0]:
# Create a UDF that takes DEP_DELAY and ARR_DELAY:
#   → DEP_DELAY <= 0              → "NOT_DELAYED"
#   → ARR_DELAY < DEP_DELAY       → "RECOVERED"
#   → ARR_DELAY > DEP_DELAY       → "WORSENED"
#   → ARR_DELAY == DEP_DELAY      → "SAME"
#
# Apply and count each category.

from pyspark.sql.functions import udf,col
from pyspark.sql.types import StringType

def categorize(dep_delay, arr_delay):
    if dep_delay is None: return "CANCELLED"
    if dep_delay <= 0: return "NOT_DELAYED"
    if arr_delay is None: return "CANCELLED"  # ← add this
    if arr_delay < dep_delay: return "RECOVERED"
    if arr_delay > dep_delay: return "WORSENED"
    return "SAME"

categorize_udf = udf(categorize, StringType())

ps33 = df.withColumn("DELAY_CATEGORY", categorize_udf(col("DEP_DELAY"), col("ARR_DELAY")))
ps33.groupBy("DELAY_CATEGORY").count().orderBy("count", ascending=False).show()

In [0]:
# Solve PS32 again using when/otherwise instead of UDF.
# Then answer:
#   → Which is faster and why?
#   → Which should you use in production?
#   → When is UDF justified?

from pyspark.sql.functions import col, when

# Implementing using native PySpark when / otherwise expressions
ps32_native = df.withColumn(
    "DELAY_CATEGORY",
    when(col("DEP_DELAY").isNull(), "CANCELLED")
    .when(col("DEP_DELAY") <= 0, "ON_TIME")
    .when(col("DEP_DELAY") <= 30, "MINOR")
    .otherwise("MAJOR")
)

# Aggregation execution
ps32_native.groupBy("DELAY_CATEGORY").count().orderBy(col("count").desc()).show()


In [0]:
# Native Expressions Win on Velocity: Native when / otherwise expressions compile directly into optimized Java bytecode inside the JVM, completely skipping the heavy serialization overhead that slows down Python UDFs.

# The Black Box Optimizer Trap: The Catalyst Optimizer treats Python UDFs as an unoptimizable black box, preventing critical execution adjustments like predicate pushdown or code generation from running across your pipelines.

# Process Switching Penalties: Python UDFs force PySpark to spin up separate worker processes and constantly pipe data back and forth between Java and Python, which heavily drains CPU and memory resources.

# The Production Standard: You must always use native PySpark functions in production to handle massive datasets efficiently, minimizing cloud compute costs and ensuring stable execution profiles.

# When UDFs Are Justified: A UDF is only acceptable when implementing complex logic that requires specialized external libraries (like scipy or numpy) or when parsing completely unstructured data streams that native relational operators cannot process.